# ViT5 Decode Tuning On Test

Notebook nay khong train lai. No load checkpoint `vit5_base_ep3_t4x2/best`, chay 5 cau hinh decode tren full test, roi xuat bang ROUGE dung cho bao cao.

Input can attach:

```text
anhnguyen0812/nlp-vit5-ep3-old-output
anhnguyen0812/nlp-vietnamese-sumarization
```


In [1]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys
import time
from zipfile import ZipFile

PROJECT_NAME = 'pretrained-summarization'
REPO_URL = 'https://github.com/Anhnguyen0812/pretrained-summarization.git'
REFRESH_REPO = True
WORKING = Path('/kaggle/working')
WORKING_REPO = WORKING / PROJECT_NAME
OUTPUT_ROOT = WORKING / 'decode_tuning_outputs'
REPORT_DIR = OUTPUT_ROOT / '_report'
DATA_DIR = Path('/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization')
TRAIN_FILE = DATA_DIR / 'train-00000-of-00001.parquet'
VALID_FILE = DATA_DIR / 'valid-00000-of-00001.parquet'
TEST_FILE = DATA_DIR / 'test-00000-of-00001.parquet'

SOURCE_RUN_ROOTS = [
    Path('/kaggle/input/nlp-vit5-ep3-old-output/summarization_outputs'),
    Path('/kaggle/input/datasets/anhnguyen0812/nlp-vit5-ep3-old-output/summarization_outputs'),
]
SOURCE_RUN_NAME = 'vit5_base_ep3_t4x2'
MAX_TEST_SAMPLES = None  # None = full test
EVAL_BATCH_SIZE = 4
OVERWRITE_EVALS = False

DECODE_CONFIGS = [
    {'name': 'beam4_lp08_max160_min50', 'num_beams': 4, 'length_penalty': 0.8, 'max_length': 160, 'min_length': 50},
    {'name': 'beam4_lp10_max160_min70', 'num_beams': 4, 'length_penalty': 1.0, 'max_length': 160, 'min_length': 70},
    {'name': 'beam4_lp12_max192_min70', 'num_beams': 4, 'length_penalty': 1.2, 'max_length': 192, 'min_length': 70},
    {'name': 'beam6_lp10_max160_min70', 'num_beams': 6, 'length_penalty': 1.0, 'max_length': 160, 'min_length': 70},
    {'name': 'beam6_lp11_max192_min70', 'num_beams': 6, 'length_penalty': 1.1, 'max_length': 192, 'min_length': 70},
]

os.chdir(WORKING)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print('CMD:', cmd, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f'Command failed with exit code {code}: {cmd}\nLast lines:\n{"".join(lines[-100:])}')
    return code

def is_repo(path):
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'vn_summarization').exists()

if REFRESH_REPO and WORKING_REPO.exists():
    shutil.rmtree(WORKING_REPO)
if not is_repo(WORKING_REPO):
    run(f'git clone --depth 1 {REPO_URL} {WORKING_REPO}', cwd=WORKING)
else:
    run('git pull --ff-only', cwd=WORKING_REPO, check=False)
repo = WORKING_REPO
run('git log --oneline -1', cwd=repo)


CMD: git clone --depth 1 https://github.com/Anhnguyen0812/pretrained-summarization.git /kaggle/working/pretrained-summarization
Cloning into '/kaggle/working/pretrained-summarization'...
CMD: git log --oneline -1
8dea660 Add ViT5 decode tuning notebook


0

In [2]:
os.chdir(repo)
run(f'{sys.executable} -m pip install -q --upgrade pip', cwd=repo)
run(f'{sys.executable} -m pip install -q -e .', cwd=repo)
run(f'{sys.executable} -m pip install -q --upgrade "transformers>=4.51.0,<5" "tokenizers>=0.22.0,<=0.23.0"', cwd=repo)
run(f'{sys.executable} -m pip check', cwd=repo, check=False)
run(f'{sys.executable} -m pip show transformers tokenizers peft accelerate | sed -n "/Name: /p;/Version: /p"', cwd=repo, check=False)
run(f"{sys.executable} -c 'import tokenizers, transformers; print(\"TRANSFORMERS\", transformers.__version__); print(\"TOKENIZERS\", tokenizers.__version__)'", cwd=repo)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
run('nvidia-smi', check=False)

for path in [TRAIN_FILE, VALID_FILE, TEST_FILE]:
    print(path, path.exists())
if not TRAIN_FILE.exists() or not VALID_FILE.exists() or not TEST_FILE.exists():
    raise FileNotFoundError('Attach dataset anhnguyen0812/nlp-vietnamese-sumarization first.')


CMD: /usr/bin/python3 -m pip install -q --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.1 MB/s eta 0:00:00
CMD: /usr/bin/python3 -m pip install -q -e .
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompa

## Locate Source Checkpoint

Source checkpoint la `vit5_base_ep3_t4x2/best`. Notebook uu tien Kaggle UI path `/kaggle/input/nlp-vit5-ep3-old-output/summarization_outputs`.


In [3]:
def find_source_run():
    for root in SOURCE_RUN_ROOTS:
        run_dir = root / SOURCE_RUN_NAME
        print('SOURCE_ROOT:', root, root.exists())
        print('SOURCE_RUN:', run_dir, run_dir.exists())
        if (run_dir / 'resolved_config.json').exists() and (run_dir / 'best' / 'model.safetensors').exists():
            return run_dir
    raise FileNotFoundError('Missing vit5_base_ep3_t4x2 with resolved_config.json and best/model.safetensors')

SOURCE_RUN_DIR = find_source_run()
SOURCE_MODEL_PATH = SOURCE_RUN_DIR / 'best'
SOURCE_CONFIG_PATH = SOURCE_RUN_DIR / 'resolved_config.json'
print('SOURCE_RUN_DIR:', SOURCE_RUN_DIR)
print('SOURCE_MODEL_PATH:', SOURCE_MODEL_PATH)


SOURCE_ROOT: /kaggle/input/nlp-vit5-ep3-old-output/summarization_outputs False
SOURCE_RUN: /kaggle/input/nlp-vit5-ep3-old-output/summarization_outputs/vit5_base_ep3_t4x2 False
SOURCE_ROOT: /kaggle/input/datasets/anhnguyen0812/nlp-vit5-ep3-old-output/summarization_outputs True
SOURCE_RUN: /kaggle/input/datasets/anhnguyen0812/nlp-vit5-ep3-old-output/summarization_outputs/vit5_base_ep3_t4x2 True
SOURCE_RUN_DIR: /kaggle/input/datasets/anhnguyen0812/nlp-vit5-ep3-old-output/summarization_outputs/vit5_base_ep3_t4x2
SOURCE_MODEL_PATH: /kaggle/input/datasets/anhnguyen0812/nlp-vit5-ep3-old-output/summarization_outputs/vit5_base_ep3_t4x2/best


In [4]:
import yaml

GENERATED_CONFIG_DIR = repo / 'configs' / '_generated_decode_tuning'
GENERATED_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

def load_json(path):
    with Path(path).open('r', encoding='utf-8') as f:
        return json.load(f)

def save_yaml(data, path):
    with Path(path).open('w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

def make_eval_config(decode_cfg):
    cfg = load_json(SOURCE_CONFIG_PATH)
    cfg = json.loads(json.dumps(cfg))
    out_dir = OUTPUT_ROOT / decode_cfg['name']
    cfg.setdefault('model', {})['name_or_path'] = str(SOURCE_MODEL_PATH)
    cfg['model']['cache_dir'] = None
    cfg.setdefault('data', {}).update({
        'train_file': str(TRAIN_FILE),
        'valid_file': str(TEST_FILE),
        'max_source_length': 768,
        'max_target_length': decode_cfg['max_length'],
        'preprocessing_num_proc': 1,
        'max_train_samples': None,
        'max_eval_samples': MAX_TEST_SAMPLES,
    })
    cfg.setdefault('training', {}).update({
        'output_dir': str(out_dir),
        'per_device_eval_batch_size': EVAL_BATCH_SIZE,
        'precision': 'fp16',
        'report_to': [],
    })
    cfg['generation'] = {
        'max_length': decode_cfg['max_length'],
        'min_length': decode_cfg['min_length'],
        'num_beams': decode_cfg['num_beams'],
        'length_penalty': decode_cfg['length_penalty'],
        'no_repeat_ngram_size': 3,
        'repetition_penalty': 1.05,
        'early_stopping': True,
    }
    config_path = GENERATED_CONFIG_DIR / f"{decode_cfg['name']}.yaml"
    save_yaml(cfg, config_path)
    return config_path, out_dir

for cfg in DECODE_CONFIGS:
    print(cfg)


{'name': 'beam4_lp08_max160_min50', 'num_beams': 4, 'length_penalty': 0.8, 'max_length': 160, 'min_length': 50}
{'name': 'beam4_lp10_max160_min70', 'num_beams': 4, 'length_penalty': 1.0, 'max_length': 160, 'min_length': 70}
{'name': 'beam4_lp12_max192_min70', 'num_beams': 4, 'length_penalty': 1.2, 'max_length': 192, 'min_length': 70}
{'name': 'beam6_lp10_max160_min70', 'num_beams': 6, 'length_penalty': 1.0, 'max_length': 160, 'min_length': 70}
{'name': 'beam6_lp11_max192_min70', 'num_beams': 6, 'length_penalty': 1.1, 'max_length': 192, 'min_length': 70}


## Run Decode Tuning

Moi config chay full test rieng va luu predictions vao `decode_tuning_outputs/<config>/predictions_test.jsonl`.


In [5]:
eval_start = time.time()
for decode_cfg in DECODE_CONFIGS:
    config_path, out_dir = make_eval_config(decode_cfg)
    metrics_path = out_dir / 'validation_metrics.json'
    if metrics_path.exists() and not OVERWRITE_EVALS:
        print('SKIP existing:', decode_cfg['name'], metrics_path)
        continue
    predictions_path = out_dir / 'predictions_test.jsonl'
    rel_config = Path(config_path).relative_to(repo).as_posix()
    cmd = (
        f'{sys.executable} -u -m vn_summarization.evaluate '
        f'--config {rel_config} '
        f'--model_path {SOURCE_MODEL_PATH} '
        f'--predictions_path {predictions_path}'
    )
    print('\n' + '=' * 100)
    print('DECODE_CONFIG:', decode_cfg)
    t0 = time.time()
    run(cmd, cwd=repo)
    print('DONE', decode_cfg['name'], 'elapsed_min=', round((time.time() - t0) / 60, 2))
print('ALL elapsed_hours=', round((time.time() - eval_start) / 3600, 3))



DECODE_CONFIG: {'name': 'beam4_lp08_max160_min50', 'num_beams': 4, 'length_penalty': 0.8, 'max_length': 160, 'min_length': 50}
CMD: /usr/bin/python3 -u -m vn_summarization.evaluate --config configs/_generated_decode_tuning/beam4_lp08_max160_min50.yaml --model_path /kaggle/input/datasets/anhnguyen0812/nlp-vit5-ep3-old-output/summarization_outputs/vit5_base_ep3_t4x2/best --predictions_path /kaggle/working/decode_tuning_outputs/beam4_lp08_max160_min50/predictions_test.jsonl
2026-06-10 19:42:50.489498: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781120570.728642      97 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781120570.794472      97 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBL

In [6]:
def pick(metrics, name):
    for key in [name, f'eval_{name}', f'test_{name}', f'predict_{name}']:
        if key in metrics:
            return metrics[key]
    for key, value in metrics.items():
        if key.endswith('_' + name):
            return value
    return ''

def to_float(value):
    try:
        return float(value)
    except Exception:
        return float('-inf')

rows = []
for decode_cfg in DECODE_CONFIGS:
    out_dir = OUTPUT_ROOT / decode_cfg['name']
    metrics_path = out_dir / 'validation_metrics.json'
    metrics = load_json(metrics_path) if metrics_path.exists() else {}
    row = {
        'run': decode_cfg['name'],
        'model': SOURCE_RUN_NAME,
        'beams': decode_cfg['num_beams'],
        'length_penalty': decode_cfg['length_penalty'],
        'max_length': decode_cfg['max_length'],
        'min_length': decode_cfg['min_length'],
        'rouge1': pick(metrics, 'rouge1'),
        'rouge2': pick(metrics, 'rouge2'),
        'rougeL': pick(metrics, 'rougeL'),
        'loss': pick(metrics, 'loss'),
        'gen_len': pick(metrics, 'gen_len'),
        'metrics_file': str(metrics_path.relative_to(WORKING)) if metrics_path.exists() else '',
        'predictions_file': str((out_dir / 'predictions_test.jsonl').relative_to(WORKING)) if (out_dir / 'predictions_test.jsonl').exists() else '',
    }
    rows.append(row)
rows = sorted(rows, key=lambda row: to_float(row.get('rougeL')), reverse=True)

columns = ['run', 'model', 'beams', 'length_penalty', 'max_length', 'min_length', 'rouge1', 'rouge2', 'rougeL', 'loss', 'gen_len', 'metrics_file', 'predictions_file']
csv_path = REPORT_DIR / 'decode_tuning_results.csv'
with csv_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    for row in rows:
        writer.writerow({col: row.get(col, '') for col in columns})

lines = ['# ViT5 Decode Tuning Results', '', '| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---'] * len(columns)) + ' |']
for row in rows:
    lines.append('| ' + ' | '.join(str(row.get(col, '')) for col in columns) + ' |')
lines.append('')
if rows:
    lines.append('Best by rougeL: ' + rows[0]['run'])
md_path = REPORT_DIR / 'decode_tuning_results.md'
md_path.write_text('\n'.join(lines), encoding='utf-8')
(REPORT_DIR / 'best_decode_config.json').write_text(json.dumps(rows[0] if rows else {}, ensure_ascii=False, indent=2), encoding='utf-8')
print(md_path.read_text(encoding='utf-8'))
print('CSV:', csv_path)
print('BEST:', REPORT_DIR / 'best_decode_config.json')


# ViT5 Decode Tuning Results

| run | model | beams | length_penalty | max_length | min_length | rouge1 | rouge2 | rougeL | loss | gen_len | metrics_file | predictions_file |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| beam4_lp10_max160_min70 | vit5_base_ep3_t4x2 | 4 | 1.0 | 160 | 70 | 74.2406 | 46.7444 | 48.8905 | 0.9785369634628296 | 150.875 | decode_tuning_outputs/beam4_lp10_max160_min70/validation_metrics.json | decode_tuning_outputs/beam4_lp10_max160_min70/predictions_test.jsonl |
| beam4_lp08_max160_min50 | vit5_base_ep3_t4x2 | 4 | 0.8 | 160 | 50 | 74.0718 | 46.5303 | 48.8577 | 0.9785369634628296 | 149.0 | decode_tuning_outputs/beam4_lp08_max160_min50/validation_metrics.json | decode_tuning_outputs/beam4_lp08_max160_min50/predictions_test.jsonl |
| beam4_lp12_max192_min70 | vit5_base_ep3_t4x2 | 4 | 1.2 | 192 | 70 | 74.205 | 46.8643 | 48.8524 | 0.9877288341522217 | 153.5179 | decode_tuning_outputs/beam4_lp12_max192_min70/validation_metrics.jso

In [7]:
zip_path = WORKING / 'decode_tuning_results.zip'
if zip_path.exists():
    zip_path.unlink()
keep_suffixes = {'.json', '.jsonl', '.csv', '.md', '.txt'}
files = [p for p in OUTPUT_ROOT.rglob('*') if p.is_file() and p.suffix in keep_suffixes]
with ZipFile(zip_path, 'w') as zf:
    for file in files:
        zf.write(file, file.relative_to(WORKING).as_posix())
print('ZIP:', zip_path)
for file in sorted(files):
    print(file)


ZIP: /kaggle/working/decode_tuning_results.zip
/kaggle/working/decode_tuning_outputs/_report/best_decode_config.json
/kaggle/working/decode_tuning_outputs/_report/decode_tuning_results.csv
/kaggle/working/decode_tuning_outputs/_report/decode_tuning_results.md
/kaggle/working/decode_tuning_outputs/beam4_lp08_max160_min50/predictions_test.jsonl
/kaggle/working/decode_tuning_outputs/beam4_lp08_max160_min50/validation_metrics.json
/kaggle/working/decode_tuning_outputs/beam4_lp10_max160_min70/predictions_test.jsonl
/kaggle/working/decode_tuning_outputs/beam4_lp10_max160_min70/validation_metrics.json
/kaggle/working/decode_tuning_outputs/beam4_lp12_max192_min70/predictions_test.jsonl
/kaggle/working/decode_tuning_outputs/beam4_lp12_max192_min70/validation_metrics.json
/kaggle/working/decode_tuning_outputs/beam6_lp10_max160_min70/predictions_test.jsonl
/kaggle/working/decode_tuning_outputs/beam6_lp10_max160_min70/validation_metrics.json
/kaggle/working/decode_tuning_outputs/beam6_lp11_max192_